# GPT and Autoregressive Models

Train a tiny causal Transformer on a repeated token sequence. Check shifted next-token targets, the causal mask, and how a vocabulary head produces logits. This local toy model is not a language model for real text.

In [ ]:
import torch
from torch import nn

torch.manual_seed(32)
torch.set_num_threads(1)
print('CPU | tiny causal next-token model')

In [ ]:
vocabulary_size, length, width = 7, 5, 8
tokens = torch.tensor([[1, 2, 3, 4, 5]])
inputs, targets = tokens[:, :-1], tokens[:, 1:]
assert inputs.tolist() == [[1, 2, 3, 4]]
assert targets.tolist() == [[2, 3, 4, 5]]
print('input tokens:', inputs.tolist(), 'next-token targets:', targets.tolist())

In [ ]:
embedding = nn.Embedding(vocabulary_size, width)
position = nn.Embedding(length, width)
layer = nn.TransformerEncoderLayer(d_model=width, nhead=2, dim_feedforward=16,
                                    batch_first=True, dropout=0.0)
backbone = nn.TransformerEncoder(layer, num_layers=1)
head = nn.Linear(width, vocabulary_size)
mask = torch.triu(torch.ones(inputs.shape[1], inputs.shape[1], dtype=torch.bool), diagonal=1)
positions = torch.arange(inputs.shape[1]).unsqueeze(0)
hidden = backbone(embedding(inputs) + position(positions), mask=mask)
logits = head(hidden)
loss = nn.functional.cross_entropy(logits.reshape(-1, vocabulary_size), targets.reshape(-1))
loss.backward()
assert logits.shape == (1, 4, vocabulary_size)
assert torch.isfinite(loss) and embedding.weight.grad is not None
print('logits:', tuple(logits.shape), '| next-token loss:', round(loss.item(), 4))
print('causal mask:')
print(mask.int())

The causal mask blocks each position from reading tokens to its right. The real GPT training objective uses very large text datasets and many optimization updates; this tiny example only checks the shifted-target mechanics.